In [ ]:
# SINGLE CELL: PSI/PSR annotation pipeline (AnnoLLM-style + Option B voting across V prompts)
# - Reads from: messages.json
# - Works on a COPY: 23_AnnoLLM.json (created once if missing)
# - Writes to disk AFTER EVERY MESSAGE (max resumability)
# - Prints an update AFTER EVERY MESSAGE
# - Exports: 23_AnnoLLM.xlsx
# - Model: gpt-5.1 (set below)
# - Step 1: Explain GOLD demos in V different explanation styles (cached)
# - Step 2: Build V developer prompts
# - Step 3: For each message, run V annotations and majority-vote labels -> write ONE final label set

import os, json, time, re, sys, shutil, uuid
from datetime import datetime
from typing import Dict, Any, List

# -----------------------------
# 0) Install/import deps
# -----------------------------
def _pip_install(pkgs: List[str]):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import pandas as pd
except Exception:
    _pip_install(["pandas", "openpyxl"])
    import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    _pip_install(["python-dotenv"])
    from dotenv import load_dotenv

try:
    import tiktoken
except Exception:
    _pip_install(["tiktoken"])
    import tiktoken

try:
    from openai import OpenAI
except Exception:
    _pip_install(["openai"])
    from openai import OpenAI


# -----------------------------
# 1) Config
# -----------------------------
ROOT_DIR = os.getcwd()

SOURCE_MESSAGES_PATH = os.path.join(ROOT_DIR, "messages.json")  # <-- read from this

RUN_BASENAME       = "23_AnnoLLM"
WORK_MESSAGES_PATH = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.json")
XLSX_OUT           = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.xlsx")

# AnnoLLM cache files
DEMOS_GOLD_PATH = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_demos_gold.json")
DEMO_RATS_PATH  = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_demo_rationales.json")

# Stats file for resumable time + usage
STATS_PATH = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_run_stats.json")

ANNOTATION_MODEL = "gpt-5.1"
EXPLAIN_MODEL    = ANNOTATION_MODEL
FIX_JSON_MODEL   = ANNOTATION_MODEL

MAX_OUTPUT_TOKENS = 2000  # per annotation call

# Pricing (per 1M tokens) — adjust if your contract differs
PRICE_INPUT_PER_1M  = 1.25
PRICE_CACHED_PER_1M = 0.125
PRICE_OUTPUT_PER_1M = 10.00

# AnnoLLM behavior flags
GENERATE_DEMO_RATIONALES_IF_MISSING = True   # Step-1 Explain runs only if no cache exists
MAX_DEMOS_FOR_PROMPT = 3                    # keep small to prevent prompt bloat

BINARY_FIELDS = [
    "DirectAddress", "Attachment", "SelfDisclose", "ReplySeeking",
    "CommRitual", "Monetary", "Backseat", "Emotes"
]
OTHER_FIELDS = ["Dimension"]
ALL_FIELDS = BINARY_FIELDS + OTHER_FIELDS + ["Notes"]

# Option B: Vote across V prompts
N_EXPL_VARIANTS = 5
VOTE_K = N_EXPL_VARIANTS

EXPLAIN_STYLES = [
    "Variant 1 (cue-first): cite the exact token(s) that trigger each label.",
    "Variant 2 (rule-first): cite the guideline rule/definition that applies, then point to the cue.",
    "Variant 3 (contrastive): briefly say what would make it NOT that label, then why it IS that label.",
    "Variant 4 (minimal): ultra-short rationales (<=10 words) but still grounded in cues.",
    "Variant 5 (mechanistic): describe how the label decision follows from the message form (addressing, directive, ritual phrase, etc.).",
]

DEBUG_STORE_VARIANT_OUTPUTS = False  # if True, store per-variant outputs in each msg (bloats JSON)


# -----------------------------
# 2) Task guidelines + Explain prompt
# -----------------------------
BASE_GUIDELINES = r"""
You are an annotation model. Your task is to label each Twitch chat message with a set of binary/ternary fields using the provided schema.
Follow definitions and decision rules exactly. Return ONLY valid JSON (no wrappers, no extra text).

OUTPUT FORMAT (STRICT)
Return ONE JSON OBJECT with keys:
DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension, Notes

Binary label fields must be exactly one of:
"Y" = present
""  = not present / leave blank

For Dimension, use exactly one of:
"P" if positive/affiliative stance toward streamer/community
"N" if negative/hostile stance toward streamer/community
No neutral/mixed category. If mixed, label based on the dominant intent (“punchline”).

WHAT YOU ARE ANNOTATING
Annotate message-level expressed cues in livestream chat from Twitch. Multi-label is allowed.
Do not infer hidden intent beyond the text and immediate conversational form.

CATEGORY DECISION RULES
A) Dimension (valence)
Set Dimension independently from other categories.
P (Positive): praise/support, gratitude, affection, encouragement, admiration; playful teasing that is clearly supportive.
N (Negative): insults, hostility, shaming, threats, aggressive blame; hate-watch enjoyment of failure.
Mixed: choose dominant intent.

B) PSI/PSR-leaning categories (multi-label)
1) Direct Address / second-person to streamer -> DirectAddress
Mark "Y" only when the message targets the streamer as an addressee:
- explicit @streamer
- streamer name used as addressee (“Emiru, …”)
- second-person pronouns clearly referring to streamer not another person (“you/your/u”) with directed statement/command
Avoid false positives:
- talking about streamer: “emiru is cracked”
- addressing chat: “you guys…”
- ambiguous “you” -> blank unless strong cue (name/@mention).

Important Point! You should keep an eye on morphs, euphemisms, and varying forms of chatting most common for Gen-Z, like "ya", "yo" etc.

2) Attachment / affection / relational warmth -> Attachment
Mark "Y" for warmth/care/pride/missing/love toward streamer or relational closeness markers.
Attachment-strengtheners:
- longitudinal persistence beyond exposure: off-stream anticipation, absence felt, “all week,” “can’t wait,” “glad you’re back”
- investment/loyalty talk: “watch every stream,” “since 2018,” “never miss a VOD,” schedule organized around streams
- social realism / real-life tie framing: “you’re like family,” “real friend,” “like a brother/sister”
- anthropomorphism/personification: “you care about us,” “she understands me,” “he worries about chat”
Avoid false positives:
- “love this game” (not streamer-directed)
- generic hype alone (“Pog”, “Lets go”) unless clearly relational.

3) Identification / self-disclosure / homophily -> SelfDisclose
Mark "Y" when viewer shares personal info or similarity claims aimed at relational connection:
- “I’m also…”, “same here…”, “as a fellow…”, "I also do that..."
- personal disclosure positioned as sharing-with-streamer (“I had a rough day, thanks for being here”)
- “I have an exam tomorrow, wish me luck” can count if framed as a bid for connection.
Strengthener:
- knowledge/narrative recall is not SelfDisclose by itself; use it to support other labels if it functions that way.

4) Reciprocity bids / reply-seeking -> ReplySeeking
Mark "Y" when seeking acknowledgment/response/recognition:
- “notice me,” “read this,” “answer please,” “remember me,” “did you see my last message,” “thanks for replying last week,” "why you not seeing me"
- Or asking them to do something ordinary like drinking water or blink for safety of eyes.
Avoid false positives:
- generic info questions not aimed at streamer unless streamer-directed by name/@mention.

5) Community ritual talk & co-creation -> CommRitual
Mark "Y" for shared rituals/inside jokes/coordinated norms tied to community:
- “spam LUL,” “copy pasta time,” “chat do the thing,” “only real ones remember…,” “as always…”
Avoid false positives:
- emotes alone ≠ ritual talk without ritual framing.

6) Monetary support actions -> Monetary
Mark "Y" for subs/donos/gifts/streaks or accompanying text:
- “subbed X months,” “gifted subs,” “dono,” “renewed,” “superchat”
Hard rule: If msgTag is "USERNOTICE" ALWAYS mark Monetary="Y".
Avoid false positives:
- unrelated money talk (“costs $60”).

ADDED CATEGORIES (behavioral / surface form)
7) Backseat guidance -> Backseat
Mark "Y" for gameplay/stream/content direction/coaching/strategy/idea:
- “go left,” “use ult,” “buy armor,” “skip cutscene,” “watch out…,” "You have to change it in the menu"
Or general questions asking regarding the topic if clearly directing streamer:
- "Hassan are they capitalists?"
Backseat can co-occur with DirectAddress, but backseating is not parasocial by itself.

8) Emotes/emoji presence -> Emotes
Mark "Y" if any Twitch-style emote tokens or unicode emojis appear, including <3, and tokens like Kappa, pogchamp, KEKW, etc.

NOTES FIELD
Write at most ~20 words explaining borderline decisions (optional). Keep it short.
"""

# --- AnnoLLM Step 1 prompt (Explain demos with GOLD labels) ---
EXPLAIN_PROMPT = r"""
You are constructing rationalized demonstrations for a Twitch chat annotation task.

You will be given:
- TASK_GUIDELINES (definitions + decision rules)
- DEMO_INPUT: {msgTag, raw_msg}
- GOLD_LABELS: the correct labels

Your job:
1) Do NOT change GOLD_LABELS.
2) Produce a concise rationale for:
   - each binary label that is "Y"
   - Dimension always
3) Do NOT add rationales for labels that are blank.
4) Output ONLY valid JSON with this exact shape:

{
  "rationale": {
    "<LabelName>": "<one short sentence>",
    ...
    "Dimension": "<one short sentence>"
  }
}

Allowed LabelName keys (use ONLY these when applicable):
- DirectAddress
- Attachment
- SelfDisclose
- ReplySeeking
- CommRitual
- Monetary
- Backseat
- Emotes
- Dimension

Rationale-writing rules:
- One short sentence per key (max ~20 words).
- Quote or point to the specific cue in the message (e.g., name mention, “what I missed?”, “LETSGO”, emoji/emote token).
- If msgTag="USERNOTICE" and Monetary="Y", the Monetary rationale MUST mention the hard rule: USERNOTICE implies Monetary.
- For Dimension, justify the stance briefly (supportive/neutral-friendly vs hostile). If the message is generic but non-hostile, treat as P.

EXAMPLES (follow this style exactly; these are illustrative, not additional demos):

Example 1
DEMO_INPUT: {"msgTag":"PRIVATEMESSAGE","raw_msg":"it’s time for us to get on that eye cream and calcium supplements"}
GOLD_LABELS: {"SelfDisclose":"Y","Dimension":"P", ... others blank}
OUTPUT:
{
  "rationale": {
    "SelfDisclose": "Shares a personal/collective circumstance about aging using “us” and self-care actions.",
    "Dimension": "Non-hostile, affiliative tone with a supportive self-care framing."
  }
}

Example 2
DEMO_INPUT: {"msgTag":"USERNOTICE","raw_msg":"just in time for more pokemon LETSGO"}
GOLD_LABELS: {"CommRitual":"Y","Monetary":"Y","Dimension":"P", ... others blank}
OUTPUT:
{
  "rationale": {
    "CommRitual": "“LETSGO” is a coordinated hype/chant typical of chat ritual framing.",
    "Monetary": "msgTag is USERNOTICE, so Monetary is forced by the hard rule regardless of content.",
    "Dimension": "Excited, supportive hype toward the stream/content."
  }
}

Example 3
DEMO_INPUT: {"msgTag":"PRIVATEMESSAGE","raw_msg":"leslie what i missed??? fusKiss"}
GOLD_LABELS: {"DirectAddress":"Y","ReplySeeking":"Y","Emotes":"Y","Dimension":"P", ... others blank}
OUTPUT:
{
  "rationale": {
    "DirectAddress": "Uses the streamer name “leslie” as an addressee.",
    "ReplySeeking": "Asks “what i missed???” which is an explicit bid for response/recap.",
    "Emotes": "Contains the emote token “fusKiss”.",
    "Dimension": "Friendly, affiliative request with positive interaction tone."
  }
}

Example 4
DEMO_INPUT: {"msgTag":"PRIVATEMESSAGE","raw_msg":"You sound very defensive for someone that’s “not narcissistic”"}
GOLD_LABELS: {"DirectAddress":"Y","Dimension":"N", ... others blank}
OUTPUT:
{
"rationale": {
"DirectAddress": "Directly addresses the addressee with second-person “You…” as a directed accusation.",
"Dimension": "Accusatory, critical tone (“defensive”, “not narcissistic”) signals a negative/hostile stance."
}
}

Example 5
DEMO_INPUT: {"msgTag":"PRIVATEMESSAGE","raw_msg":"Fuslie, please play the game and practice it so when you play the nuzlocke you have experience"}
GOLD_LABELS: {"DirectAddress":"Y","Backseat":"Y","Dimension":"P", ... others blank}
OUTPUT:
{
"rationale": {
"DirectAddress": "Uses the streamer name “Fuslie” as an explicit addressee.",
"Backseat": "Gives directive coaching (“please play… practice… so… you have experience”) about how to prepare/play.",
"Dimension": "Constructive guidance framed politely (“please”) indicates a non-hostile, supportive intent."
}
}

Now produce rationales for the provided DEMO_INPUT and GOLD_LABELS.
Return ONLY valid JSON. No code fences. No extra text.
"""

# -----------------------------
# 3) Env + client
# -----------------------------
load_dotenv(os.path.join(ROOT_DIR, ".env"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists with OPENAI_API_KEY=...")

client = OpenAI()


# -----------------------------
# 4) Utility helpers
# -----------------------------
def now_ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def clip_text(s: str, max_chars: int = 220) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    if len(s) <= max_chars:
        return s
    return s[:max_chars - 1] + "…"

def short_preview(s: str, n: int = 90) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    if len(s) <= n:
        return s
    return s[:n - 1] + "…"

def safe_write_json(path: str, obj: Any, retries: int = 10, sleep: float = 0.05):
    directory = os.path.dirname(path)
    os.makedirs(directory, exist_ok=True)
    tmp = f"{path}.{uuid.uuid4().hex}.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    for _ in range(retries):
        try:
            shutil.move(tmp, path)
            return
        except PermissionError:
            time.sleep(sleep)
    raise PermissionError(f"Could not replace {path} after {retries} retries")

def load_json_if_exists(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def ensure_working_copy():
    if not os.path.exists(SOURCE_MESSAGES_PATH):
        raise FileNotFoundError(f"messages.json not found at: {SOURCE_MESSAGES_PATH}")
    if not os.path.exists(WORK_MESSAGES_PATH):
        shutil.copy2(SOURCE_MESSAGES_PATH, WORK_MESSAGES_PATH)
        print(f"[{now_ts()}] Created working copy: {WORK_MESSAGES_PATH}")
    else:
        print(f"[{now_ts()}] Using existing working copy (resume): {WORK_MESSAGES_PATH}")

def load_messages() -> List[Dict[str, Any]]:
    with open(WORK_MESSAGES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Working messages file must be a JSON array of message objects.")
    return data

def is_annotated(msg: Dict[str, Any]) -> bool:
    return str(msg.get("Annotated", "")).strip().lower() == "true"

def mark_annotated(msg: Dict[str, Any]):
    msg["Annotated"] = "true"

def ensure_fields(msg: Dict[str, Any]):
    for k in BINARY_FIELDS + OTHER_FIELDS:
        msg.setdefault(k, "")
    for k in ["raw_msg", "msgTag", "streamer", "Annotated", "Notes"]:
        msg.setdefault(k, "")
    if msg["Annotated"] == "":
        msg["Annotated"] = "False"

def get_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-4.1")
    except Exception:
        return tiktoken.get_encoding("o200k_base")

ENC = get_encoding()

def usage_breakdown(resp) -> Dict[str, Any]:
    u = getattr(resp, "usage", None)
    if not u:
        return {}
    try:
        return u if isinstance(u, dict) else u.model_dump()
    except Exception:
        try:
            return dict(u)
        except Exception:
            return {"raw_usage": str(u)}

def enforce_binary(v: Any) -> str:
    return "Y" if str(v).strip().upper() == "Y" else ""

def normalize_dim(v: Any) -> str:
    s = str(v).strip().upper()
    return "P" if s == "P" else ("N" if s == "N" else "")

def clip_notes(text: str, max_words: int = 20) -> str:
    t = re.sub(r"\s+", " ", (text or "").strip())
    if not t:
        return ""
    words = t.split(" ")
    if len(words) <= max_words:
        return t
    return " ".join(words[:max_words]).rstrip() + "…"

def call_with_retries(kwargs, tries=6, base_sleep=1.5):
    last_err = None
    for attempt in range(1, tries + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as e:
            last_err = e
            sleep = base_sleep * (2 ** (attempt - 1))
            print(f"[{now_ts()}]   ERROR (attempt {attempt}/{tries}): {e}. Sleeping {sleep:.1f}s then retry...")
            time.sleep(sleep)
    raise last_err

def get_output_text(resp) -> str:
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    out = getattr(resp, "output", None) or []
    for item in out:
        if isinstance(item, dict) and item.get("type") == "message":
            for c in (item.get("content") or []):
                if isinstance(c, dict) and c.get("type") in ("output_text", "text"):
                    return c.get("text", "") or ""
    return ""

def try_repair_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    t = t.strip()
    m_obj = re.search(r"\{.*\}", t, flags=re.DOTALL)
    m_arr = re.search(r"\[.*\]", t, flags=re.DOTALL)
    if m_arr and (not m_obj or len(m_arr.group(0)) >= len(m_obj.group(0))):
        t = m_arr.group(0)
    elif m_obj:
        t = m_obj.group(0)
    t = t.replace("\t", " ")
    t = re.sub(r",\s*([}\]])", r"\1", t)
    return t.strip()

def reformat_to_json_only(bad_text: str) -> str:
    fix_prompt = (
        "You will be given text that is intended to be JSON but is invalid.\n"
        "Rewrite it into VALID JSON ONLY.\n"
        "Do not change meanings or labels. Do not omit any keys.\n"
        "Output ONLY valid JSON.\n\n"
        "TEXT:\n" + bad_text
    )
    r = client.responses.create(
        model=FIX_JSON_MODEL,
        input=[{"role": "user", "content": fix_prompt}],
        max_output_tokens=400,
        reasoning={"effort":"low"},
    )
    return (get_output_text(r) or "").strip()

def parse_json_object_generic(resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")
    t = try_repair_json(text)
    try:
        obj = json.loads(t)
    except Exception:
        fixed = reformat_to_json_only(text)
        obj = json.loads(try_repair_json(fixed))
    if isinstance(obj, list):
        if not obj or not isinstance(obj[0], dict):
            raise RuntimeError(f"Expected list[dict], got {type(obj)}.")
        obj = obj[0]
    if not isinstance(obj, dict):
        raise RuntimeError(f"Expected JSON object, got {type(obj)}.")
    return obj

def parse_single_json(resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")
    t = try_repair_json(text)
    obj = None
    try:
        obj = json.loads(t)
    except Exception:
        fixed = reformat_to_json_only(text)
        obj = json.loads(try_repair_json(fixed))

    # unwrap common shapes
    if isinstance(obj, dict) and "items" in obj and isinstance(obj["items"], list):
        obj = obj["items"]
    if isinstance(obj, list):
        if not obj:
            raise RuntimeError("Parsed JSON array empty.")
        if not isinstance(obj[0], dict):
            raise RuntimeError(f"Expected list[dict], got list[{type(obj[0])}]")
        obj = obj[0]
    if not isinstance(obj, dict):
        raise RuntimeError(f"Expected dict, got {type(obj)}")

    for k in ALL_FIELDS:
        obj.setdefault(k, "")
    return obj

def estimate_cost_usd(total_input: int, total_cached: int, total_output: int) -> float:
    cached = max(0, int(total_cached or 0))
    inp = max(0, int(total_input or 0))
    non_cached = max(0, inp - cached)
    cost = 0.0
    cost += (non_cached / 1_000_000.0) * PRICE_INPUT_PER_1M
    cost += (cached     / 1_000_000.0) * PRICE_CACHED_PER_1M
    cost += (max(0, int(total_output or 0)) / 1_000_000.0) * PRICE_OUTPUT_PER_1M
    return cost

def load_stats() -> Dict[str, Any]:
    stats = load_json_if_exists(STATS_PATH, default={})
    if not isinstance(stats, dict):
        stats = {}
    stats.setdefault("accumulated_seconds", 0.0)
    stats.setdefault("runs", 0)
    stats.setdefault("total_input_tokens", 0)
    stats.setdefault("total_output_tokens", 0)
    stats.setdefault("total_cached_tokens", 0)
    stats.setdefault("last_run_started_at", "")
    stats.setdefault("last_run_ended_at", "")
    return stats

def save_stats(stats: Dict[str, Any]):
    safe_write_json(STATS_PATH, stats)


# -----------------------------
# 5) Demos (gold) + Step 1 rationales (V variants)
# -----------------------------


def default_demos_gold() -> List[Dict[str, Any]]:
    # UPDATED: replace the old placeholder demos with these 7
    # (matches your provided labels/notes; msgTag set to PRIVATEMESSAGE except the USERNOTICE one)
    return [
        {
            "name": "D1_eyecream_calcium_selfdisclose",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "it’s time for us to get on that eye cream and calcium supplements",
            },
            "gold": {
                "DirectAddress": "",
                "Attachment": "",
                "SelfDisclose": "Y",
                "ReplySeeking": "",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
        {
            "name": "D2_time_for_bed_none",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "its time for bed",
            },
            "gold": {
                "DirectAddress": "",
                "Attachment": "",
                "SelfDisclose": "",
                "ReplySeeking": "",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
        {
            "name": "D3_pokemon_letsgo_commritual_usernotice_monetary",
            "input": {
                "msgTag": "USERNOTICE",  # per your note: USERNOTICE => Monetary forced
                "raw_msg": "just in time for more pokemon LETSGO",
            },
            "gold": {
                "DirectAddress": "",
                "Attachment": "",
                "SelfDisclose": "",
                "ReplySeeking": "",
                "CommRitual": "Y",
                "Monetary": "Y",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
        {
            "name": "D4_about_talk_sabotage_none",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "leslie is playing to sabotage om",
            },
            "gold": {
                "DirectAddress": "",
                "Attachment": "",
                "SelfDisclose": "",
                "ReplySeeking": "",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
        {
            "name": "D5_pokeslayer_attachment_only",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "Leslie the PokeSlayer PokPikachu",
            },
            "gold": {
                "DirectAddress": "",
                "Attachment": "Y",
                "SelfDisclose": "",
                "ReplySeeking": "",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
        {
            "name": "D6_what_i_missed_directaddress_replyseeking_emotes",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "leslie what i missed??? fusKiss",
            },
            "gold": {
                "DirectAddress": "Y",
                "Attachment": "",
                "SelfDisclose": "",
                "ReplySeeking": "Y",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "Y",
                "Dimension": "P",
            },
        },
        {
            "name": "D7_shirts_hitting_directaddress_attachment_replyseeking",
            "input": {
                "msgTag": "PRIVATEMESSAGE",
                "raw_msg": "Leslie why do your shirts always be hitting lol",
            },
            "gold": {
                "DirectAddress": "Y",
                "Attachment": "Y",
                "SelfDisclose": "",
                "ReplySeeking": "Y",
                "CommRitual": "",
                "Monetary": "",
                "Backseat": "",
                "Emotes": "",
                "Dimension": "P",
            },
        },
    ]

def ensure_demos_gold_file() -> List[Dict[str, Any]]:
    demos = load_json_if_exists(DEMOS_GOLD_PATH, default=None)
    if isinstance(demos, list) and demos:
        return demos
    demos = default_demos_gold()
    safe_write_json(DEMOS_GOLD_PATH, demos)
    print(f"[{now_ts()}] Wrote default gold demos: {DEMOS_GOLD_PATH}")
    return demos

def explain_prompt_with_style(style_text: str) -> str:
    return EXPLAIN_PROMPT.strip() + "\n\nSTYLE_CONSTRAINT:\n" + style_text.strip() + "\n"

def call_explain_prompt_variant(demo_input: Dict[str, Any], gold: Dict[str, Any], variant_id: int) -> Dict[str, Any]:
    payload = {
        "TASK_GUIDELINES": BASE_GUIDELINES.strip(),
        "DEMO_INPUT": demo_input,
        "GOLD_LABELS": gold
    }
    kwargs = dict(
        model=EXPLAIN_MODEL,
        input=[
            {"role":"developer", "content": explain_prompt_with_style(EXPLAIN_STYLES[variant_id-1])},
            {"role":"user", "content": json.dumps(payload, ensure_ascii=False)}
        ],
        max_output_tokens=700,
        reasoning={"effort":"low"},
    )
    resp = call_with_retries(kwargs)
    obj = parse_json_object_generic(resp)

    rationale = obj.get("rationale", {})
    if not isinstance(rationale, dict):
        rationale = {}

    cleaned = {}
    for k, v in gold.items():
        if k == "Dimension":
            cleaned["Dimension"] = clip_text(str(rationale.get("Dimension","")).strip() or "Dominant stance determines valence.", 140)
        elif k in BINARY_FIELDS and str(v).strip().upper() == "Y":
            cleaned[k] = clip_text(str(rationale.get(k,"")).strip() or "Text supports the labeled cue.", 140)

    return {"rationale": cleaned}

def ensure_demo_rationales_variants(demos_gold: List[Dict[str, Any]], n_variants: int) -> Dict[int, List[Dict[str, Any]]]:
    cache = load_json_if_exists(DEMO_RATS_PATH, default=None)

    # load if valid
    if isinstance(cache, dict) and "variants" in cache and isinstance(cache["variants"], dict):
        try:
            out = {int(k): v for k, v in cache["variants"].items()}
            if cache.get("n_variants") == n_variants and all(isinstance(out.get(i), list) for i in range(1, n_variants+1)):
                # basic completeness check: same number of demos
                if all(len(out[i]) >= len(demos_gold) for i in range(1, n_variants+1)):
                    print(f"[{now_ts()}] Loaded cached demo rationales variants: {DEMO_RATS_PATH}")
                    return out
        except Exception:
            pass

    if not GENERATE_DEMO_RATIONALES_IF_MISSING:
        print(f"[{now_ts()}] Demo rationales missing and generation disabled; proceeding with empty variants.")
        return {i: [] for i in range(1, n_variants+1)}

    print(f"[{now_ts()}] Generating demo rationales for {n_variants} explanation variants...")
    variants: Dict[int, List[Dict[str, Any]]] = {}

    for vid in range(1, n_variants + 1):
        rats: List[Dict[str, Any]] = []
        for i, d in enumerate(demos_gold, 1):
            name = d.get("name", f"demo_{i}")
            demo_input = d.get("input", {})
            gold = d.get("gold", {})

            explained = call_explain_prompt_variant(demo_input, gold, variant_id=vid)
            rats.append({
                "name": name,
                "input": demo_input,
                "gold": gold,
                "rationale": explained.get("rationale", {})
            })

            # incremental save so it's resumable even mid-variant
            variants[vid] = rats
            safe_write_json(DEMO_RATS_PATH, {
                "n_variants": n_variants,
                "variants": {str(k): v for k, v in variants.items()}
            })
            print(f"[{now_ts()}]   variant {vid}/{n_variants} explained {i}/{len(demos_gold)}: {name}")

        variants[vid] = rats

    print(f"[{now_ts()}] Saved demo rationales variants: {DEMO_RATS_PATH}")
    return variants


# -----------------------------
# 6) Build V developer prompts (same demo subset across variants)
# -----------------------------
def format_rationale_inline(r: Dict[str, str]) -> str:
    order = ["DirectAddress","Attachment","SelfDisclose","ReplySeeking","CommRitual","Monetary","Backseat","Emotes","Dimension"]
    parts = []
    for k in order:
        if k in r and str(r.get(k,"")).strip():
            parts.append(f"{k}: {r[k]}")
    return " | ".join(parts) if parts else ""

def build_rationalized_demos_block(rats: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, r in enumerate(rats, 1):
        demo_input = r.get("input", {})
        gold = r.get("gold", {})
        rationale = r.get("rationale", {}) or {}
        blocks.append(
            f"### Demonstration {i}\n"
            f"INPUT_MESSAGE:\n{json.dumps(demo_input, ensure_ascii=False)}\n"
            f"GOLD_OUTPUT_JSON:\n{json.dumps(gold, ensure_ascii=False)}\n"
            f"EXPLANATION (why these labels):\n{format_rationale_inline(rationale) or '(no rationale)'}\n"
        )
    return "\n".join(blocks) if blocks else "### Demonstration\n(No rationalized demos available)\n"

def build_anno_llm_developer_prompt(rationalized_demos_block: str) -> str:
    return (
        BASE_GUIDELINES.strip()
        + "\n\nRATIONALIZED DEMONSTRATIONS (use as guidance; do not copy labels blindly):\n"
        + rationalized_demos_block.strip()
        + "\n\nReturn ONLY the JSON object. No code fences. No extra keys."
    )

def demo_active_labels(gold: Dict[str, Any]) -> set:
    return {k for k in BINARY_FIELDS if str(gold.get(k,"")).strip().upper() == "Y"}

def is_hard_negative_demo(gold: Dict[str, Any]) -> bool:
    return len(demo_active_labels(gold)) == 0

def select_demo_subset_for_prompt(rats: List[Dict[str, Any]], max_demos: int = 8) -> List[Dict[str, Any]]:
    if not rats:
        return []

    pool_labels = set()
    for r in rats:
        pool_labels |= demo_active_labels(r.get("gold", {}))

    selected = []
    covered = set()
    remaining = rats[:]

    # greedy set cover
    while remaining and len(selected) < max_demos and covered != pool_labels:
        best = None
        best_gain = -1
        for r in remaining:
            gain = len(demo_active_labels(r.get("gold", {})) - covered)
            if gain > best_gain:
                best_gain = gain
                best = r
        if best is None or best_gain <= 0:
            break
        selected.append(best)
        covered |= demo_active_labels(best.get("gold", {}))
        remaining = [r for r in remaining if r is not best]

    # ensure one hard negative if exists
    if len(selected) < max_demos and not any(is_hard_negative_demo(r.get("gold", {})) for r in selected):
        negs = [r for r in rats if is_hard_negative_demo(r.get("gold", {}))]
        if negs:
            negs.sort(key=lambda r: len((r.get("input", {}).get("raw_msg") or "").strip()))
            selected.append(negs[0])

    # fill remaining slots
    if len(selected) < max_demos:
        leftover = [r for r in rats if r not in selected]
        leftover.sort(key=lambda r: -len(demo_active_labels(r.get("gold", {}))))
        selected.extend(leftover[: max(0, max_demos - len(selected))])

    return selected[:max_demos]


# -----------------------------
# 7) Voting
# -----------------------------
def majority_vote_binary(outputs: List[Dict[str, Any]], field: str) -> str:
    need = len(outputs) // 2 + 1
    ys = sum(1 for o in outputs if str(o.get(field, "")).strip().upper() == "Y")
    return "Y" if ys >= need else ""

def majority_vote_dimension(outputs: List[Dict[str, Any]]) -> str:
    need = len(outputs) // 2 + 1
    dims = [normalize_dim(o.get("Dimension","")) for o in outputs]
    p = sum(1 for d in dims if d == "P")
    n = sum(1 for d in dims if d == "N")
    if p >= need: return "P"
    if n >= need: return "N"
    for d in dims:
        if d in ("P","N"):
            return d
    return "P"

def pick_notes(outputs: List[Dict[str, Any]], voted: Dict[str, str]) -> str:
    for o in outputs:
        ok = True
        for f in BINARY_FIELDS:
            if enforce_binary(o.get(f,"")) != voted.get(f,""):
                ok = False
                break
        if ok and normalize_dim(o.get("Dimension","")) != voted.get("Dimension",""):
            ok = False
        if ok:
            note = clip_notes(o.get("Notes",""), max_words=20)
            if note:
                return note
    for o in outputs:
        note = clip_notes(o.get("Notes",""), max_words=20)
        if note:
            return note
    return ""


# -----------------------------
# 8) Setup working copy + load data + stats + build prompts
# -----------------------------
ensure_working_copy()
messages = load_messages()
for m in messages:
    ensure_fields(m)

total = len(messages)
remaining = sum(1 for m in messages if not is_annotated(m))
print(f"[{now_ts()}] Loaded {total} messages. Remaining unannotated: {remaining}")

demos_gold = ensure_demos_gold_file()
demo_rats_by_variant = ensure_demo_rationales_variants(demos_gold, N_EXPL_VARIANTS)

# pick SAME demo subset for all variants (based on variant 1)
rats_v1 = demo_rats_by_variant.get(1, [])
sel_v1 = select_demo_subset_for_prompt(rats_v1, max_demos=MAX_DEMOS_FOR_PROMPT)
sel_names = [r.get("name") for r in sel_v1 if r.get("name")]

if not sel_names:
    # fallback: just take first MAX_DEMOS from variant 1
    sel_names = [r.get("name") for r in rats_v1[:MAX_DEMOS_FOR_PROMPT] if r.get("name")]

DEVELOPER_PROMPTS: Dict[int, str] = {}
for vid in range(1, N_EXPL_VARIANTS + 1):
    rats = demo_rats_by_variant.get(vid, [])
    # keep only selected demos, in the same order as sel_names
    by_name = {r.get("name"): r for r in rats if r.get("name") in set(sel_names)}
    ordered = [by_name[n] for n in sel_names if n in by_name]
    block = build_rationalized_demos_block(ordered)
    DEVELOPER_PROMPTS[vid] = build_anno_llm_developer_prompt(block)

stats = load_stats()
stats["runs"] += 1
stats["last_run_started_at"] = now_ts()
save_stats(stats)
run_start = time.time()


# -----------------------------
# 9) Main loop (per-message; V calls; vote; write+print every message)
# -----------------------------
newly_annotated = 0

for idx, msg in enumerate(messages):
    if is_annotated(msg):
        continue

    raw_msg = msg.get("raw_msg", "") or ""
    msgTag  = msg.get("msgTag", "") or ""

    # if empty, mark and persist
    if not raw_msg.strip() and not msgTag.strip():
        mark_annotated(msg)
        newly_annotated += 1

        safe_write_json(WORK_MESSAGES_PATH, messages)
        elapsed = time.time() - run_start
        stats["accumulated_seconds"] += elapsed
        run_start = time.time()
        stats["last_run_ended_at"] = now_ts()
        save_stats(stats)

        print(f"[{now_ts()}] #{idx} EMPTY -> marked Annotated=true | newly_annotated={newly_annotated}")
        continue

    user_payload = {"msgTag": msgTag, "raw_msg": raw_msg}
    user_content = (
        "OUTPUT ONLY the final JSON object with keys:\n"
        "DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension, Notes.\n"
        "Use only \"Y\" or \"\" for binary fields.\n"
        "Return ONLY valid JSON.\n\nINPUT_MESSAGE:\n"
        + json.dumps(user_payload, ensure_ascii=False)
    )
    user_message = {"role": "user", "content": user_content}

    variant_outputs = []
    sum_in_tok = 0
    sum_out_tok = 0
    sum_cached_tok = 0
    sum_latency = 0.0

    for vid in range(1, VOTE_K + 1):
        kwargs_v = dict(
            model=ANNOTATION_MODEL,
            input=[
                {"role": "developer", "content": DEVELOPER_PROMPTS[vid]},
                user_message
            ],
            max_output_tokens=MAX_OUTPUT_TOKENS,
            reasoning={"effort":"low"},
        )

        t0 = time.time()
        resp_v = call_with_retries(kwargs_v)
        lat_v = time.time() - t0
        sum_latency += lat_v

        ud = usage_breakdown(resp_v)
        in_tok  = int(ud.get("input_tokens") or 0)
        out_tok = int(ud.get("output_tokens") or 0)
        cached_tok = 0
        try:
            cached_tok = int((ud.get("input_tokens_details") or {}).get("cached_tokens") or 0)
        except Exception:
            cached_tok = 0

        sum_in_tok += in_tok
        sum_out_tok += out_tok
        sum_cached_tok += cached_tok

        out_v = parse_single_json(resp_v)
        variant_outputs.append(out_v)

    # Update aggregate token stats (single message, V calls)
    stats["total_input_tokens"]  += sum_in_tok
    stats["total_output_tokens"] += sum_out_tok
    stats["total_cached_tokens"] += sum_cached_tok

    # Majority vote labels
    voted = {}
    for f in BINARY_FIELDS:
        voted[f] = majority_vote_binary(variant_outputs, f)
    voted["Dimension"] = majority_vote_dimension(variant_outputs)
    voted["Notes"] = pick_notes(variant_outputs, voted)

    # Apply to message
    for f in BINARY_FIELDS:
        msg[f] = voted[f]

    # Hard rule: USERNOTICE => Monetary=Y (your task definition)
    if str(msgTag).strip().upper() == "USERNOTICE":
        msg["Monetary"] = "Y"

    msg["Dimension"] = voted["Dimension"]
    msg["Notes"] = voted["Notes"]

    if DEBUG_STORE_VARIANT_OUTPUTS:
        msg["_variant_outputs"] = variant_outputs  # heavy, debugging only

    mark_annotated(msg)
    newly_annotated += 1

    # Write JSON after every message
    safe_write_json(WORK_MESSAGES_PATH, messages)

    # Update resumable time after every message
    elapsed_segment = time.time() - run_start
    stats["accumulated_seconds"] += elapsed_segment
    run_start = time.time()
    stats["last_run_ended_at"] = now_ts()
    save_stats(stats)

    total_done = sum(1 for m in messages if is_annotated(m))
    cost_now = estimate_cost_usd(stats["total_input_tokens"], stats["total_cached_tokens"], stats["total_output_tokens"])

    print(
        f"[{now_ts()}] #{idx} streamer='{(msg.get('streamer') or '').strip()}' tag='{msgTag.strip()}' "
        f"msg='{short_preview(raw_msg, 110)}'\n"
        f"  -> DirectAddress={msg['DirectAddress'] or '-'} Attachment={msg['Attachment'] or '-'} "
        f"SelfDisclose={msg['SelfDisclose'] or '-'} ReplySeeking={msg['ReplySeeking'] or '-'} "
        f"CommRitual={msg['CommRitual'] or '-'} Monetary={msg['Monetary'] or '-'} "
        f"Backseat={msg['Backseat'] or '-'} Emotes={msg['Emotes'] or '-'} "
        f"Dimension={msg['Dimension'] or '-'} Notes='{msg.get('Notes','')}'\n"
        f"  usage (V={VOTE_K}): in={sum_in_tok} out={sum_out_tok} cached_in={sum_cached_tok} | latency_total={sum_latency:.2f}s\n"
        f"  totals: annotated={total_done}/{total} | newly_annotated_this_run={newly_annotated} | est_cost_total=${cost_now:.6f}\n"
    )


# -----------------------------
# 10) Export XLSX
# -----------------------------
df = pd.DataFrame(messages)
front = ["streamer", "msgTag", "raw_msg", "Annotated"] + BINARY_FIELDS + ["Dimension", "Notes"]
cols = front + [c for c in df.columns if c not in front]
df = df[cols]
df.to_excel(XLSX_OUT, index=False)


# -----------------------------
# 11) Final summary
# -----------------------------
remaining = sum(1 for m in messages if not is_annotated(m))
total_input  = stats["total_input_tokens"]
total_output = stats["total_output_tokens"]
total_cached = stats["total_cached_tokens"]
total_cost = estimate_cost_usd(total_input, total_cached, total_output)

print(f"\n[{now_ts()}] DONE.")
print(f"[{now_ts()}] Total messages={total} | Newly annotated this run={newly_annotated} | Remaining={remaining}")
print(f"[{now_ts()}] Working JSON: {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] XLSX: {XLSX_OUT}")
print(f"[{now_ts()}] Resumable time (accumulated): {stats['accumulated_seconds']:.2f} seconds")
print(f"[{now_ts()}] Tokens total: input={total_input}, cached_input={total_cached}, output={total_output}")
print(f"[{now_ts()}] Estimated total cost ({ANNOTATION_MODEL}): ${total_cost:.6f}")
print(f"[{now_ts()}] Run stats file: {STATS_PATH}")
print(f"[{now_ts()}] Gold demos file: {DEMOS_GOLD_PATH}")
print(f"[{now_ts()}] Demo rationales cache: {DEMO_RATS_PATH}")
